Importation des bibliothèques:

Dans cette première cellule, nous importons les bibliothèques nécessaires :

numpy et pandas : pour la manipulation de données.
pathlib.Path : pour gérer les chemins de fichiers de manière plus flexible.
tqdm : pour afficher une barre de progression lors des boucles.
warnings : pour supprimer les avertissements afin de garder la sortie plus propre.
Ensuite, nous importons plusieurs fonctions de scikit-learn :

LogisticRegression : un modèle de régression logistique pour la classification.
StandardScaler : pour standardiser les données (centrer / réduire).
classification_report, confusion_matrix, balanced_accuracy_score, f1_score : métriques d’évaluation.
StratifiedKFold, cross_val_predict, GridSearchCV : pour la validation croisée et la recherche d’hyperparamètres.
SelectKBest et f_classif : pour sélectionner les meilleures variables (features) selon un test statistique (ANOVA F-value).
Pipeline : pour enchaîner plusieurs étapes de transformation et de modélisation.

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
warnings.filterwarnings("ignore")


Définition de la fonction extract_enhanced_features


Cette fonction a pour objectif d’extraire des features (caractéristiques) supplémentaires à partir des données d’images satellitaires.

Les images sont supposées avoir la forme :

X \in \text{(n_samples, n_months, n_channels, height, width)}
où :

n_samples est le nombre d’échantillons (zones ou parcelles).
n_months est le nombre de mois (ou de pas de temps).
n_channels est le nombre de canaux (ex. bandes spectrales).
height et width sont les dimensions spatiales de l’image (en pixels).
Pour chaque échantillon (chaque image), on fait :

Reshape pour faciliter le calcul : on passe de 
(
𝑛
_
𝑚
𝑜
𝑛
𝑡
ℎ
𝑠
,
𝑛
_
𝑐
ℎ
𝑎
𝑛
𝑛
𝑒
𝑙
𝑠
,
ℎ
𝑒
𝑖
𝑔
ℎ
𝑡
,
𝑤
𝑖
𝑑
𝑡
ℎ
)
(n_months,n_channels,height,width) à 
(
𝑛
_
𝑚
𝑜
𝑛
𝑡
ℎ
𝑠
,
𝑛
_
𝑐
ℎ
𝑎
𝑛
𝑛
𝑒
𝑙
𝑠
,
ℎ
𝑒
𝑖
𝑔
ℎ
𝑡
×
𝑤
𝑖
𝑑
𝑡
ℎ
)
(n_months,n_channels,height×width).
On calcule différentes statistiques (moyenne, écart-type, min, max) pour chaque canal et pour chaque mois en aplatisant la partie spatiale.
On calcule un indice NDVI-like ( 
𝑁
𝐼
𝑅
−
𝑅
𝐸
𝐷
𝑁
𝐼
𝑅
+
𝑅
𝐸
𝐷
NIR+RED
NIR−RED
​
  ) pour chaque mois, en supposant que :
Le nir (near-infrared) est le canal 0.
Le red (bande rouge) est le canal 1.
On concatène toutes ces statistiques dans un seul vecteur de taille fixe par échantillon (130 dans l’exemple).


In [5]:
# Fonction pour extraire les features des images satellites
def extract_enhanced_features(X):
    """
    Extraction avancée de caractéristiques :
    Pour chaque échantillon, calcule :
        - Moyenne, Écart-Type, Minimum, Maximum pour chaque canal et chaque mois
        - Indice NDVI pour chaque mois : (B08 - B04) / (B08 + B04)
    Retourne un tableau de forme (n_samples, n_feats).
    """
    # Dimensions des données : (n_samples, n_months, n_channels, height, width)
    # n_samples : Nombre d'échantillons (champs agricoles)
    # n_months : Nombre de mois (séquence temporelle)
    # n_channels : Nombre de canaux spectraux (ex. : NIR, RED)
    # height, width : Dimensions spatiales des images
    n_samples, n_months, n_channels, height, width = X.shape
    feature_list = []  # Liste pour stocker les vecteurs de caractéristiques

    # Boucle sur chaque échantillon pour extraire les caractéristiques
    for i in tqdm(range(n_samples), desc="Extracting Features"):
        sample = X[i]  # Récupère un échantillon : (n_months, n_channels, height, width)

        # Réorganiser l'échantillon en (n_months, n_channels, height*width)
        # Cela aplatit les dimensions spatiales pour simplifier les calculs
        sample_reshaped = sample.reshape(n_months, n_channels, -1)

        # Calcul des caractéristiques statistiques pour chaque canal et chaque mois
        mean = sample_reshaped.mean(axis=2)    # Moyenne par canal : (n_months, n_channels)
        std = sample_reshaped.std(axis=2)      # Écart-type par canal : (n_months, n_channels)
        min_ = sample_reshaped.min(axis=2)     # Minimum par canal : (n_months, n_channels)
        max_ = sample_reshaped.max(axis=2)     # Maximum par canal : (n_months, n_channels)

        # Calcul de l'indice NDVI pour chaque mois
        # Hypothèse : canal 0 = NIR, canal 1 = RED
        nir = mean[:, 0]  # Moyenne du canal NIR
        red = mean[:, 1]  # Moyenne du canal RED
        ndvi = np.where(
            (nir + red) != 0,          # Évite la division par zéro
            (nir - red) / (nir + red), # Calcul de l'indice NDVI
            0                          # Valeur par défaut pour éviter les erreurs
        )  # Résultat : (n_months,)

        # Combiner les caractéristiques statistiques et le NDVI dans un seul vecteur
        features = np.concatenate([
            mean.flatten(),  # Moyenne aplatie : (n_months * n_channels)
            std.flatten(),   # Écart-type aplati
            min_.flatten(),  # Minimum aplati
            max_.flatten(),  # Maximum aplati
            ndvi             # NDVI (n_months)
        ])  # Taille totale : (n_months * n_channels) * 4 + n_months

        # Ajouter le vecteur de caractéristiques à la liste
        feature_list.append(features)

    # Convertir la liste de vecteurs en tableau NumPy
    X_features = np.array(feature_list, dtype=np.float32)  # Format float32 pour optimiser la mémoire
    return X_features  # Retourne les caractéristiques extraites


Configuration de l’état aléatoire et chargement des données
Explications (Markdown) :

Nous fixons la graine aléatoire (RANDOM_STATE = 42) pour rendre les résultats reproductibles (important pour la validation croisée).
Nous utilisons Path pour définir le chemin vers le dossier contenant les données.
Nous chargeons les tableaux numpy :
X_train, Y_train : images et labels d’entraînement.
X_test, Y_test : images et labels de test.
On affiche la forme des tableaux chargés (nombre d’échantillons, etc.) pour vérifier que tout est conforme aux attentes.

In [7]:
# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load data paths
base_path = Path('data')
X_train = np.load(base_path / 'train_images.npy').astype(np.float32)
Y_train = np.load(base_path / 'train_labels.npy')
X_test = np.load(base_path / 'test_public_images.npy').astype(np.float32)
Y_test = np.load(base_path / 'test_public_labels.npy')

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_test shape:  {X_test.shape},  Y_test shape: {Y_test.shape}")


X_train shape: (10000, 10, 3, 32, 32), Y_train shape: (10000,)
X_test shape:  (2500, 10, 3, 32, 32),  Y_test shape: (2500,)


Extraction des features avancées
Explications (Markdown) :
Nous utilisons la fonction extract_enhanced_features définie précédemment pour extraire les features des données d’entraînement et de test. Ensuite, on affiche la forme des matrices de caractéristiques résultantes.

In [8]:
print("\nExtracting enhanced features...")
X_train_feats = extract_enhanced_features(X_train)  # shape: (n_train, 130)
X_test_feats = extract_enhanced_features(X_test)    # shape: (n_test, 130)

print(f"Feature shape (train): {X_train_feats.shape}")
print(f"Feature shape (test) : {X_test_feats.shape}")



Extracting enhanced features...


Extracting Features: 100%|██████████| 2500/2500 [00:00<00:00, 11133.25it/s]

Feature shape (train): (10000, 130)
Feature shape (test) : (2500, 130)


Définition du pipeline
Explications (Markdown) :
Dans le pipeline, nous chaînons plusieurs étapes :

StandardScaler : pour centrer et réduire les variables (moyenne = 0, écart-type = 1).
SelectKBest : pour sélectionner les 
𝑘
k meilleures features selon un test statistique (ici, ANOVA F-value).
LogisticRegression : notre classifieur (modèle de machine learning).
Nous précisons dans la régression logistique :

penalty='l2' : régularisation L2.
max_iter=1000 : nombre maximum d’itérations.
solver='lbfgs' : algorithme d’optimisation.
class_weight='balanced' : permet de prendre en compte le déséquilibre de classes.
n_jobs=-1 : utilise tous les cœurs disponibles pour accélérer le calcul.
random_state : pour la reproductibilité.

In [9]:
print("\nScaling features and selecting best features...")

# Définition du pipeline de traitement
pipeline = Pipeline([
    # Étape 1 : Normalisation des caractéristiques
    ('scaler', StandardScaler()),  
    # Utilise StandardScaler pour centrer et réduire les données (moyenne = 0, écart-type = 1)
    # Permet d'homogénéiser les échelles des caractéristiques pour améliorer la convergence des modèles.

    # Étape 2 : Sélection des meilleures caractéristiques
    ('feature_selection', SelectKBest(score_func=f_classif, k=50)),  
    # Sélectionne les 50 meilleures caractéristiques basées sur un test ANOVA (f_classif).
    # Ce test mesure la corrélation entre chaque caractéristique et la variable cible.
    # Réduit la dimensionnalité tout en conservant les informations les plus pertinentes.

    # Étape 3 : Modèle de régression logistique
    ('clf', LogisticRegression(
        penalty='l2',                # Ajout de régularisation L2 pour éviter le surapprentissage
        max_iter=1000,               # Fixe un maximum de 1000 itérations pour garantir la convergence
        solver='lbfgs',              # Utilise l'algorithme "lbfgs", efficace pour les problèmes multiclasse
        class_weight='balanced',     # Gère le déséquilibre des classes en ajustant les poids automatiquement
        n_jobs=-1,                   # Utilise tous les cœurs disponibles pour paralléliser les calculs
        random_state=RANDOM_STATE    # Fixe une graine aléatoire pour assurer la reproductibilité
    ))
])



Scaling features and selecting best features...


Définition de la grille d’hyperparamètres et initialisation de la validation croisée
Explications (Markdown) :

Nous définissons un dictionnaire param_grid pour chercher les meilleures valeurs :

clf__C : coefficient de régularisation (inverse de la force de régularisation).
feature_selection__k : nombre de caractéristiques à sélectionner (50, 70, ou toutes).
Nous utilisons une validation croisée stratifiée ( StratifiedKFold ) pour garder la même proportion de classes dans chaque sous-échantillon.

In [10]:
# Réglage des hyperparamètres pour la régression logistique
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],         # Force de la régularisation (inverse de la pénalité)
    # C contrôle la régularisation du modèle :
    # - Valeurs basses (ex. : 0.01) appliquent une forte régularisation, ce qui réduit les coefficients du modèle.
    # - Valeurs élevées (ex. : 100) appliquent une faible régularisation, ce qui rend le modèle plus flexible.
    # Choix : Tester plusieurs valeurs pour trouver un équilibre entre biais et variance.

    'feature_selection__k': [30, 50, 70, 'all']  # Nombre de meilleures caractéristiques à sélectionner
    # Ce paramètre détermine combien de caractéristiques seront conservées par SelectKBest.
    # - 'all' signifie conserver toutes les caractéristiques.
    # Choix : Tester différents niveaux de réduction de la dimensionnalité pour identifier celui qui optimise les performances.
}

# Création d'une validation croisée avec Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# Stratified K-Fold divise les données en 5 sous-ensembles (folds) tout en préservant la distribution des classes.
# Cela garantit que chaque fold contient approximativement la même proportion de chaque classe que dans l'ensemble complet.
# - shuffle=True : Mélange les données avant de les diviser, ce qui améliore la randomisation.
# - random_state : Fixe une graine pour assurer la reproductibilité des splits.



Recherche d’hyperparamètres (GridSearchCV)
Explications (Markdown) :

On utilise GridSearchCV pour tester toutes les combinaisons d’hyperparamètres spécifiées dans param_grid.
scoring='f1_macro' : on optimise le F1-score macro (gère mieux le déséquilibre).
cv=skf : on fait la validation croisée avec StratifiedKFold.
n_jobs=-1 : exécute en parallèle sur tous les cœurs.
verbose=1 : affiche un peu plus d’informations pendant l’entraînement.


In [11]:
print("\nStarting Grid Search for hyperparameter tuning...")

# Initialisation de la recherche par grille (Grid Search) pour le réglage des hyperparamètres
grid_search = GridSearchCV(
    estimator=pipeline,          # Le pipeline défini précédemment contenant le scaler, SelectKBest et LogisticRegression
    param_grid=param_grid,       # La grille d'hyperparamètres à tester
    scoring='f1_macro',          # Métrique utilisée pour évaluer les modèles
    # f1_macro : Moyenne F1 non pondérée sur toutes les classes, favorisant une bonne performance même sur les classes minoritaires.

    cv=skf,                      # Validation croisée avec Stratified K-Fold (préserve la distribution des classes dans chaque fold)
    n_jobs=-1,                   # Utilise tous les cœurs disponibles pour accélérer les calculs
    verbose=1                    # Niveau de verbosité pour afficher la progression de la recherche
)

# Entraînement de Grid Search
grid_search.fit(X_train_feats, Y_train)  
# Le GridSearchCV :
# - Entraîne le pipeline pour chaque combinaison d'hyperparamètres dans la grille.
# - Évalue chaque combinaison avec une validation croisée sur 5 folds (définis par StratifiedKFold).
# - Identifie les hyperparamètres donnant les meilleures performances selon la métrique spécifiée (ici f1_macro).

# Afficher les meilleurs hyperparamètres trouvés
print(f"\nBest Hyperparameters: {grid_search.best_params_}")
# Affiche les valeurs optimales de 'clf__C' et 'feature_selection__k' après évaluation de toutes les combinaisons.

# Afficher le meilleur score F1 macro obtenu lors de la validation croisée
print(f"Best Cross-Validated F1 Macro Score: {grid_search.best_score_:.4f}")
# Affiche le score moyen obtenu sur les 5 folds avec les meilleurs hyperparamètres.



Starting Grid Search for hyperparameter tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Hyperparameters: {'clf__C': 10, 'feature_selection__k': 70}
Best Cross-Validated F1 Macro Score: 0.6073


Évaluation en validation croisée avec le meilleur modèle
Explications (Markdown) :

On récupère le meilleur pipeline trouvé par la grid search : grid_search.best_estimator_.
On effectue des prédictions en validation croisée avec cross_val_predict afin de mesurer les performances globales sur l’ensemble du jeu d’entraînement.
On calcule :
Balanced Accuracy : métrique adaptée en cas de classes déséquilibrées.
F1-Score Macro : F1 calculé de manière équitable sur toutes les classes.
Matrice de confusion.
Classification report (précision, rappel, F1).

In [12]:
best_pipeline = grid_search.best_estimator_

print("\n=== Cross-Validation Evaluation ===")
y_pred_cv = cross_val_predict(best_pipeline, X_train_feats, Y_train, cv=skf, n_jobs=-1)

balanced_acc_cv = balanced_accuracy_score(Y_train, y_pred_cv)
macro_f1_cv = f1_score(Y_train, y_pred_cv, average='macro')
print(f"Cross-Validated Balanced Accuracy: {balanced_acc_cv:.4f}")
print(f"Cross-Validated Macro F1-Score: {macro_f1_cv:.4f}")

cm_cv = confusion_matrix(Y_train, y_pred_cv)
print("Cross-Validation Confusion Matrix:\n", cm_cv)

report_cv = classification_report(Y_train, y_pred_cv, zero_division=0)
print("Cross-Validation Classification Report:\n", report_cv)



=== Cross-Validation Evaluation ===
Cross-Validated Balanced Accuracy: 0.6888
Cross-Validated Macro F1-Score: 0.6081
Cross-Validation Confusion Matrix:
 [[  34    1    1    0    0    0    0    0    0    0    1    2    0    0
     0    0    1    0    0    9]
 [   1   26    0    0    0    0    1    0    0    0    0    7    0   10
     0    0    0    1    0    1]
 [  84    4 1853    1    5    4    7    4    3  185   81   37    4    3
     3    0    2    1    3  180]
 [   0    0    0  253    0    4    0    1    0    0    0    0    0    0
     0    6    0    4    4    0]
 [   3    0    3    0  682    0    2    1    0   20    0    3    0    0
     1    0    7    0    1    4]
 [   0    0    0    2    0   61    7    0    0    1    0    0    0    0
     0    0    0    0    1    0]
 [   4    2    1    0    2   16  170    1    2   11    2    2   10    0
     5    0   34    2    0    3]
 [   2    1    0    3    5    0    4  536  152    5    4    0    2    0
     0    0    4   17   17    5]
 [   0

Entraînement sur l’ensemble du jeu d’entraînement et test final
Explications (Markdown) :

À ce stade, on ré-entraîne le meilleur pipeline sur toutes les données d’entraînement.
On teste ensuite sur le jeu de test.
On calcule et affiche les mêmes métriques de performance : Balanced Accuracy, Macro F1-Score, matrice de confusion et classification report sur le test.

In [13]:
print("\n=== Training on Entire Training Set ===")
best_pipeline.fit(X_train_feats, Y_train)

print("\n=== Evaluation on the Test Set ===")
y_pred_test = best_pipeline.predict(X_test_feats)

balanced_acc_test = balanced_accuracy_score(Y_test, y_pred_test)
macro_f1_test = f1_score(Y_test, y_pred_test, average='macro')
print(f"Test Balanced Accuracy: {balanced_acc_test:.4f}")
print(f"Test Macro F1-Score: {macro_f1_test:.4f}")

cm_test = confusion_matrix(Y_test, y_pred_test)
print("Test Confusion Matrix:\n", cm_test)

report_test = classification_report(Y_test, y_pred_test, zero_division=0)
print("Test Classification Report:\n", report_test)



=== Training on Entire Training Set ===

=== Evaluation on the Test Set ===
Test Balanced Accuracy: 0.5227
Test Macro F1-Score: 0.4780
Test Confusion Matrix:
 [[  2   0   0   0   1   0   0   0   0   0   5   2   0   0   0   0   0   0
    0   1]
 [  0   0   0   0   0   0   0   0   0   0   0   4   0   0   0   0   0   0
    0   0]
 [ 12   0 275   0   3   0   4   1   0  81 263  16   3   0   0   2   0   0
    1 120]
 [  0   0   0 105   0   0   0   0   0   0   0   0   0   0   0   3   0   0
    0   0]
 [  0   0   0   0 212   0   1   0   0   5   0   1   0   0   1   0   2   0
    3   0]
 [  0   0   0   0   0  28   8   0   1   1   1   0   0   0   0   0   0   0
    0   0]
 [  0   0   0   0   1   5  16   0   1   2   0   0   1   0   0   0   9   0
    0   1]
 [  0   2   0   0   0   4   2  29  30   1   1   0   0   0   3   0   1   7
    4   0]
 [  0   3   0   0   2   2   1  17  58   3   0   0   1   0   1   3   2  18
    5   0]
 [  0   0   2   0   1   0   0   1   0   1   3   0   0   0   0   0   2   0
 